<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [2]</a>'.</span>

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [2]:
df_train = pd.read_csv('/data/train.csv')
df_metadata = pd.read_csv('/data/metaData.csv')
df_train.sample(5)

FileNotFoundError: [Errno 2] No such file or directory: '/data/train.csv'

In [ ]:
info_df = pd.DataFrame({
    "column": df_train.columns,
    "non_null_count": df_train.notnull().sum().values,
    
    "dtype": df_train.dtypes.values
})

info_df

In [ ]:
# COLUMN TYPE GROUPING


# TARGET + IDENTIFIER


ID_COL = ["event_id"]
TARGET_COLS = ["time_to_hit_hours", "event"]


# CATEGORICAL COUNT FEATURES


categorical_count = [
    "num_perimeters_0_5h"
]


# TIME / TEMPORAL FEATURES


time_features = [
    "dt_first_last_0_5h",
    "event_start_hour",
    "event_start_dayofweek",
    "event_start_month"
]


# BINARY / LOW CARDINALITY FEATURES


categorical = [
    "low_temporal_resolution_0_5h"
]


# CONTINUOUS FEATURES


continuous = [
    "area_first_ha",
    "area_growth_abs_0_5h",
    "area_growth_rel_0_5h",
    "area_growth_rate_ha_per_h",
    "log1p_area_first",
    "log1p_growth",
    "log_area_ratio_0_5h",
    "relative_growth_0_5h",
    "radial_growth_m",
    "radial_growth_rate_m_per_h",
    "centroid_displacement_m",
    "centroid_speed_m_per_h",
    "spread_bearing_deg",
    "spread_bearing_sin",
    "spread_bearing_cos",
    "dist_min_ci_0_5h",
    "dist_std_ci_0_5h",
    "dist_change_ci_0_5h",
    "dist_slope_ci_0_5h",
    "closing_speed_m_per_h",
    "closing_speed_abs_m_per_h",
    "projected_advance_m",
    "dist_accel_m_per_h2",
    "dist_fit_r2_0_5h",
    "alignment_cos",
    "alignment_abs",
    "cross_track_component",
    "along_track_speed"
]


# ALL FEATURES


ALL_FEATURES = continuous + categorical + categorical_count + time_features

print("Continuous:", len(continuous))
print("Categorical:", len(categorical))
print("Categorical Count:", len(categorical_count))
print("Time Features:", len(time_features))
print("Total Features:", len(ALL_FEATURES))


In [ ]:
df_train_cleaned = df_train.copy()

In [ ]:
df_train['low_temporal_resolution_0_5h'].unique()

In [ ]:
df_train['num_perimeters_0_5h'].unique()

# Temporal Unique Values

In [ ]:
for i in time_features:
    print(f"\n{i}")
    print(df_train[i].unique())


### Validation of Temporal Features

The temporal metadata variables were checked against their expected ranges based on the dataset documentation.

- `dt_first_last_0_5h` values fall within the expected 0–5 hour range.
- `event_start_hour` values are within the valid 0–23 hour range.
- `event_start_dayofweek` correctly spans values from 0 (Monday) to 6 (Sunday).
- `event_start_month` values fall within the valid range of 1–12, although only months 1–9 appear in the dataset.

Overall, all temporal features are consistent with the metadata definitions and require no additional preprocessing.

# Continuous Unique Values

In [ ]:
for i in continuous:
    print(f"\n{i}")
    print(df_train[i].unique())

# Replacing Wrong Data

Tiny Negative Noise → Replace with 0

area_growth_abs_0_5h<br>
area_growth_rel_0_5h<br>
area_growth_rate_ha_per_h<br>
log_area_ratio_0_5h<br>
radial_growth_m<br>
radial_growth_rate_m_per_h<br>
dist_slope_ci_0_5h<br>
dist_accel_m_per_h2

In [ ]:
noise_cols = [
    "area_growth_abs_0_5h",
    "area_growth_rel_0_5h",
    "area_growth_rate_ha_per_h",
    "log_area_ratio_0_5h",
    "radial_growth_m",
    "radial_growth_rate_m_per_h"
]

for col in noise_cols:
    df_train_cleaned[col] = df_train_cleaned[col].clip(lower=0)

# Checking Dupliates

In [ ]:
df_metadata.duplicated().sum()

# Missing Values

In [ ]:
df_train.isnull().sum()

In [ ]:
# update continuous list
valid_continuous = [col for col in continuous if col in df_train_cleaned.columns]

# boxplots
for i in valid_continuous:
    plt.figure(figsize=(6,4))
    sns.boxplot(x=df_train_cleaned[i])
    plt.title(i)
    plt.show()

### Outlier Handling

Boxplots revealed the presence of several extreme values in the dataset. 
However, these values represent realistic wildfire behavior such as very large fires or fires located far from evacuation zones. 

Since the dataset is small and tree-based models are robust to outliers, these observations were retained to preserve important information about extreme wildfire events.


`HINDI`<br>
dataset me jo outliers dikh rahe hain
wo actual wildfire events ho sakte hain
isliye unhe remove nahi kiya gaya

sirf measurement noise values ko clean kiya gaya
taaki real extreme fires dataset me remain karein

In [ ]:
df_train_cleaned.to_csv("../data/train_cleaned.csv", index=False)

### Saving Cleaned Dataset

A cleaned copy of the training dataset was created to preserve the original data. 
All preprocessing steps were applied to this new dataset, which was then saved for use in the modeling phase.